# Exercise: Data Preparation and Cleaning

#### Note:
You do not need to submit the completed notebook. However, the open-ended questions at the end should be discussed in the Slack channel, and **they will count toward your class participation credit**.

---

# Load the Data

Import Pandas and read the `'spotify_songs_sample.csv'` file into a DataFrame, setting `track_id` column as the index. Then display the first 7 rows of the DataFrame.

In [1]:
import pandas as pd
df = pd.read_csv('spotify_songs_sample.csv', index_col='track_id')
df.head(5)

,rank,track_name,artist_names,album_name,album_id,popularity,duration,explicit,release_date,album_type,copies
track_id,,,,,,,,,,,
3aSWXU6owkZeVhh94XxEWO,90,30 For 30 (with Kendrick Lamar),SZA|Kendrick Lamar,SOS Deluxe: LANA,3VQkNrG74QPY4rHBPoyZYZ,89.0,4:38,1.0,2024-12-20,album,2.0
5Z75FvRFiultmPFWHx5jQ7,47,7 Dias,Gabito Ballesteros|Tito Double P,7 Dias,5BoeqKVMi7D4S6U38rWjS4,90.0,3:00,1.0,2025-01-30,single,3.0
0FDzzruyVECATHXKHFs9eJ,84,A Sky Full of Stars,Coldplay,Ghost Stories,2G4AUqfwxcV1UdQjm2ouYr,89.0,4:27,0.0,2014-05-16,album,861.0
3FjK86616FbluOfTxNK2gY,67,ANXIETY (feat. Doechii),Sleepy Hallow|Doechii,Boy Meets World,2Qq4N5lYtsZspF2QFLKcbY,89.0,2:28,1.0,2023-09-15,album,1.0
5vNRhkKd0yEAg8suGBpjeY,8,APT.,ROSÉ|Bruno Mars,APT.,2IYQwwgxgOIn7t3iF6ufFD,95.0,2:49,0.0,2024-10-18,single,1160.0


# Data Exploration

- Which columns have missing values and and how many are missing in each?

In [2]:
print(df.info())

<class 'pandas.core.frame.DataFrame'>
Index: 100 entries, 3aSWXU6owkZeVhh94XxEWO to 0aB0v4027ukVziUGwVGYpG
Data columns (total 11 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   rank          100 non-null    int64  
 1   track_name    100 non-null    object 
 2   artist_names  100 non-null    object 
 3   album_name    95 non-null     object 
 4   album_id      100 non-null    object 
 5   popularity    93 non-null     float64
 6   duration      100 non-null    object 
 7   explicit      95 non-null     float64
 8   release_date  100 non-null    object 
 9   album_type    100 non-null    object 
 10  copies        85 non-null     float64
dtypes: float64(3), int64(1), object(7)
memory usage: 9.4+ KB
None


In [3]:
df.isna().sum()

rank             0
track_name       0
artist_names     0
album_name       5
album_id         0
popularity       7
duration         0
explicit         5
release_date     0
album_type       0
copies          15
dtype: int64

## Change `album_type` data type
Numbers are easier to process than strings. Since `album_type` has a finite set of options, it is common practice to convert such categorical data into numbers. In the case of a binary category, converting it to boolean is also a commobn approach.

- How many unique values are in the `album_type` column and what are those values?

In [4]:
df.album_type.unique()

array(['album', 'single'], dtype=object)

In [5]:
print(f"There are {df.album_type.nunique()} unique albulm types: {df.album_type.unique()}")

There are 2 unique albulm types: ['album' 'single']


As seen in the previouse question, there are only two `album_type` values - `album` and `single`. We can convert them into boolean. 

- Set `album` to `False` and `single` to `True`.

In [6]:
df['album_type'] = df['album_type'].map({'album': False, 'single': True})
df.dtypes

rank              int64
track_name       object
artist_names     object
album_name       object
album_id         object
popularity      float64
duration         object
explicit        float64
release_date     object
album_type         bool
copies          float64
dtype: object

# Let's handle the missing values

### `explicit`

- Remove all the data entries where the `explicit` column has missing values.

In [7]:
df = df.dropna(subset=['explicit'])

In [8]:
# checking
df.isna().sum()

rank             0
track_name       0
artist_names     0
album_name       5
album_id         0
popularity       7
duration         0
explicit         0
release_date     0
album_type       0
copies          15
dtype: int64

### `copies`
- Replace all the missing values in the `copies` column with the median of the column.

In [9]:
median_copies = df['copies'].median()
df['copies'] = df['copies'].fillna(median_copies)

In [10]:
# checking
df.isna().sum()

rank            0
track_name      0
artist_names    0
album_name      5
album_id        0
popularity      7
duration        0
explicit        0
release_date    0
album_type      0
copies          0
dtype: int64

### `album_name`
We observe that some values in the `album_name` column are missing, while the `album_id` column contains no missing values. This implies that even when an album name is missing, the corresponding album can still be uniquely identified by its `album_id`. If another row shares the same `album_id` and has a non-missing `album_name`, both rows refer to the same album.

Therefore, we can safely replace missing `album_name` values with the album name from another entry that has the same `album_id`. 

- Identify the `album_id` values corresponding to entries with missing `album_name`.

In [11]:
df[df['album_name'].isna()]['album_id'].unique()

array(['5K79FLRUCSysQnVESLcTdb', '3iPSVi54hsacKKl1xIR2eH',
       '3OxfaVgvTxUTy7276t7SPU', '7aJuG4TFXa2hmE4z1yxc3n'], dtype=object)

- Check whether each of these `album_id` values appears in another entry with a non-missing `album_name`.
- Fill the missing `album_name` values using the album name from the corresponding entry with the same `album_id`.

In [12]:
for album_id in df[df['album_name'].isna()]['album_id'].unique():
    album_name = df[(df['album_id'] == album_id) & (df['album_name'].notna())]['album_name'].values
    if len(album_name) > 0:
        df.loc[(df['album_id'] == album_id) & (df['album_name'].isna()), 'album_name'] = album_name[0]

In [13]:
# # Using groupby(), this task can be done without using loop
# df['album_name'] = df.groupby('album_id')['album_name'].transform('first')

# # or alternatively
# df['album_name'] = (
#     df.groupby('album_id')['album_name']
#       .transform(lambda x: x.ffill().bfill())
# )

In [14]:
# checking
df.isna().sum()

rank            0
track_name      0
artist_names    0
album_name      0
album_id        0
popularity      7
duration        0
explicit        0
release_date    0
album_type      0
copies          0
dtype: int64

### `artist_names`
The `artist_names` column contains multiple artists stored as a single string, separated by a delimiter (`|`). Split this column so that each entry becomes a Python list of artist names.

In [15]:
df['artist_names'] = df['artist_names'].str.split('|')

In [16]:
df

,rank,track_name,artist_names,album_name,album_id,popularity,duration,explicit,release_date,album_type,copies
track_id,,,,,,,,,,,
3aSWXU6owkZeVhh94XxEWO,90,30 For 30 (with Kendrick Lamar),"[SZA, Kendrick Lamar]",SOS Deluxe: LANA,3VQkNrG74QPY4rHBPoyZYZ,89.0,4:38,1.0,2024-12-20,False,2.0
5Z75FvRFiultmPFWHx5jQ7,47,7 Dias,"[Gabito Ballesteros, Tito Double P]",7 Dias,5BoeqKVMi7D4S6U38rWjS4,90.0,3:00,1.0,2025-01-30,True,3.0
0FDzzruyVECATHXKHFs9eJ,84,A Sky Full of Stars,[Coldplay],Ghost Stories,2G4AUqfwxcV1UdQjm2ouYr,89.0,4:27,0.0,2014-05-16,False,861.0
3FjK86616FbluOfTxNK2gY,67,ANXIETY (feat. Doechii),"[Sleepy Hallow, Doechii]",Boy Meets World,2Qq4N5lYtsZspF2QFLKcbY,89.0,2:28,1.0,2023-09-15,False,1.0
5vNRhkKd0yEAg8suGBpjeY,8,APT.,"[ROSÉ, Bruno Mars]",APT.,2IYQwwgxgOIn7t3iF6ufFD,95.0,2:49,0.0,2024-10-18,True,1160.0
...,...,...,...,...,...,...,...,...,...,...,...
0fK7ie6XwGxQTIkpFoWkd1,22,like JENNIE,[JENNIE],Ruby,1vWMw6pu3err6qqZzI3RhH,93.0,2:03,1.0,2025-03-07,False,5.0
2CGNAOSuO1MEFCbBRgUzjd,12,luther (with sza),"[Kendrick Lamar, SZA]",GNX,1Ss0ArMRr91m83mOgRBjSZ,94.0,2:57,0.0,2024-11-21,False,2.0
7ricDBUakN3N0YkkKE8Obu,49,mi refe,"[Beéle, Ovy On The Drums]",Mi Refe,78NyvN1k9wNAf3NyZlxtxM,90.0,2:39,0.0,2024-12-05,True,3.0


### `popularity`
- Can you find the relation between the columns `rank` and `popularity`?

Hint:
```python 
     df[['rank', 'popularity']].sort_values(by=['rank', 'popularity'], ascending=True)
```
This shows only the `rank` and `popularity` columns and sorts them first by `rank` and then by `popularity`.

- Do you notice any pattern there?

In [17]:
df[['rank', 'popularity']].sort_values(by=['rank', 'popularity'], ascending=True)

,rank,popularity
track_id,,
2plbrEY59IikOBgBGLjaoe,1,100.0
3sK8wGT43QFpWrvNQsrQya,2,98.0
6dOtVTDdiauQNBQEDOtlAB,3,98.0
0zirWZTcXBBwGsevrsIpvT,4,97.0
2lTm559tuIvatlT1u0JYG2,5,96.0
...,...,...
4OLT65TRsOx3Iv1TlCcQVb,96,88.0
7BqBn9nzAq8spo5e7cZ0dJ,97,NaN
6iOndD4OFo7GkaDypWQIou,98,88.0


It appears that `rank` and `popularity` are corelated. With this relationship in mind, which of the follwoing options would make the most sense to do to resolve the missing values in the `popularity` column?

<input type="radio" name="popularity_fill"> Fill with the `mean popularity over the entire dataset` <br>
<input type="radio" name="popularity_fill"> Fill with the `median popularity over the entire dataset`<br>
<input type="radio" name="popularity_fill"> Fill with the `mean popularity of the tracks ranked immediately before and after the missing entry`<br>

-----

- Code the option you have chosen. 

In [18]:
# 1. Sort by rank
df = df.sort_values('rank')

# 2. Fill the NaN values using local average (interpolation)
df['popularity'] = df['popularity'].interpolate().ffill().bfill()

# # Above code is equivalent to:
# df['popularity'] = df['popularity'].interpolate(limit_direction='both')

# ### Solution manually handling edge cases.
# ### This would be less efficient than using interpolate().
# # 2. Get the popularity of the rank before and the rank after
# prev_popularity = df['popularity'].shift(1)
# next_popularity = df['popularity'].shift(-1)

# # 3. Apply the specific condition for the last row: 
# # "1 rank after is the same as 1 rank before"
# next_popularity.iloc[-1] = prev_popularity.iloc[-1]

# # Also, need to handle the first row of prev_popularity?
# # "1 rank before is set to 1 rank after
# prev_popularity.iloc[0] = next_popularity.iloc[0]

# # 4. Calculate the mean of the neighbors
# fill_values = (prev_popularity + next_popularity) / 2

# # 5. Fill the NaN values
# df['popularity'] = df['popularity'].fillna(fill_values)

- Change the type of the `popularity` column to `int`.

In [19]:
df['popularity'] = df['popularity'].astype(int)

In [20]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 95 entries, 2plbrEY59IikOBgBGLjaoe to 4DnrAI8WyUY6gkOwl8GlPN
Data columns (total 11 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   rank          95 non-null     int64  
 1   track_name    95 non-null     object 
 2   artist_names  95 non-null     object 
 3   album_name    95 non-null     object 
 4   album_id      95 non-null     object 
 5   popularity    95 non-null     int64  
 6   duration      95 non-null     object 
 7   explicit      95 non-null     float64
 8   release_date  95 non-null     object 
 9   album_type    95 non-null     bool   
 10  copies        95 non-null     float64
dtypes: bool(1), float64(2), int64(2), object(6)
memory usage: 8.3+ KB


<div style="background-color: #ffffcc; padding: 15px; border-left: 5px solid #ffd700; color: #333;"> 
    <strong>Discuss about your choice </strong>in the Slack ics-604-applied-data channel. <br> 
    Explain why you think your choice makes sense.
</div>

<div style="background-color: #ffffcc; padding: 15px; border-left: 5px solid #ffd700; color: #333;"> 
<strong>Thinking Question:</strong> 
If you have chosen option 3, how would you handle it if the first or last row had 'NaN' value?
</div>